In [1]:

# CLn


import os, random, math
import numpy as np
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import torchvision, torchvision.transforms as transforms
import timm


LEARNING_RATE    = 5e-4     
LABEL_SMOOTHING  = 0.1     
DROP_PATH_RATE   = 0.0    
KD_TEMPERATURE   = 3.0    
DECAY_RATE       = 0.10     
TASK_WEIGHT      = 0.5     
DISTILL_WEIGHT   = 0.5       
EMA_DECAY        = 0.999     

SEED            = 67
BATCH_SIZE      = 64
EPOCHS          = 10
#LEARNING_RATE   = 8e-4
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_DIR       = '/kaggle/input/datasets/melikechan/cifar100/cifar100/train'
TEST_DIR        = '/kaggle/input/datasets/melikechan/cifar100/cifar100/test'
MODEL_PATH      = '/kaggle/input/models/totallyapoorv/resnetoncifar100/pytorch/default/1/resnetall.pth'
DATA_SCALES     = ['10%', '25%', '50%', '100%']
SCALE_MAP       = {'10%': 0.10, '25%': 0.25, '50%': 0.50, '100%': 1.0}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic = True

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

full_trainset = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=transform_train)
testset       = torchvision.datasets.ImageFolder(TEST_DIR,  transform=transform_test)
all_indices   = list(range(len(full_trainset)))
random.shuffle(all_indices)

criterion_kl = nn.KLDivLoss(reduction='batchmean')

def soft_kd_loss(s_logits, t_logits, T=KD_TEMPERATURE):
    return criterion_kl(F.log_softmax(s_logits/T, dim=1),
                        F.softmax(t_logits/T, dim=1)) * (T**2)

def cosine_sem_loss(t_feat, s_feat, proj):
    return 1.0 - F.cosine_similarity(t_feat, proj(s_feat), dim=1).mean()

def make_scheduler(opt, total_epochs, base_lr, min_lr=1e-6):
    def lr_fn(epoch):
        p = epoch / max(total_epochs - 1, 1)
        c = 0.5 * (1 + math.cos(math.pi * p))
        return (min_lr / base_lr) + (1 - min_lr / base_lr) * c
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

results_p1 = {}

for DATA_SCALE in DATA_SCALES:
    print(f"\n{'='*65}\n  P1 | {DATA_SCALE}\n{'='*65}")
    CKPT = f"p1_{DATA_SCALE}.pth"

    n = int(len(full_trainset) * SCALE_MAP[DATA_SCALE])
    trainloader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(full_trainset, all_indices[:n]),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    testloader = torch.utils.data.DataLoader(
        testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print(f"  {n} images | {len(trainloader)} batches/epoch")

    teacher = torchvision.models.resnet18(weights=None)
    teacher.fc = nn.Linear(teacher.fc.in_features, 100)
    teacher.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    teacher = teacher.to(DEVICE).eval()

    student = timm.create_model('deit_tiny_distilled_patch16_224',
                                 pretrained=False, num_classes=100,
                                 drop_path_rate=DROP_PATH_RATE)
    student.set_distilled_training(True)
    student = student.to(DEVICE)

    proj_head = nn.Linear(192, 512).to(DEVICE)
    criterion_ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    cache = {}
    teacher.avgpool.register_forward_hook(
        lambda m, i, o: cache.update({'t_pool': o.view(o.size(0), -1)}))
    student.head_dist.register_forward_hook(
        lambda m, i, o: cache.update({'s_dist': i[0]}))

    params = list(student.parameters()) + list(proj_head.parameters())
    opt    = optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.05)

    start_epoch = 0; best_acc = 0.0
    if os.path.exists(CKPT):
        ck = torch.load(CKPT)
        student.load_state_dict(ck['model']); proj_head.load_state_dict(ck['proj'])
        opt.load_state_dict(ck['opt'])
        start_epoch = ck['epoch'] + 1; best_acc = ck['best_acc']
        print(f"  Resumed from epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS):
        student.train(); proj_head.train()
        run_loss = 0.0
        sw = math.exp(-DECAY_RATE * epoch)

        for inputs, targets in trainloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            opt.zero_grad()
            with torch.no_grad(): t_out = teacher(inputs)
            out = student(inputs)
            s_cls, s_dist = out if isinstance(out, tuple) else (out, out)
            loss = (TASK_WEIGHT    * criterion_ce(s_cls, targets)
                  + DISTILL_WEIGHT * soft_kd_loss(s_dist, t_out)
                  + sw             * cosine_sem_loss(cache['t_pool'], cache['s_dist'], proj_head))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            run_loss += loss.item()

        student.eval()
        correct = total = 0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                out = student(inputs)
                if isinstance(out, tuple): out = (out[0] + out[1]) / 2
                correct += out.max(1)[1].eq(targets).sum().item(); total += targets.size(0)
        val_acc = 100. * correct / total
        if val_acc > best_acc: best_acc = val_acc
        print(f"  Ep {epoch+1:2d} | Loss {run_loss/len(trainloader):.4f} | "
              f"Val {val_acc:.2f}% | SW {sw:.3f} | LR {opt.param_groups[0]['lr']:.5f}")
        torch.save({'epoch': epoch, 'model': student.state_dict(), 'proj': proj_head.state_dict(),
                    'opt': opt.state_dict(), 'best_acc': best_acc}, CKPT)

    results_p1[DATA_SCALE] = best_acc
    print(f" Best: {best_acc:.2f}%")

print(f"\n{'='*65}\nP1 SUMMARY\n{'='*65}")
for s, a in results_p1.items(): print(f"  {s:>5}: {a:.2f}%")


  P1 | 10%
  5000 images | 79 batches/epoch
  Ep  1 | Loss 7.4963 | Val 3.44% | SW 1.000 | LR 0.00050
  Ep  2 | Loss 6.8979 | Val 4.85% | SW 0.905 | LR 0.00050
  Ep  3 | Loss 6.4723 | Val 7.08% | SW 0.819 | LR 0.00050
  Ep  4 | Loss 6.1810 | Val 8.76% | SW 0.741 | LR 0.00050
  Ep  5 | Loss 5.9684 | Val 9.20% | SW 0.670 | LR 0.00050
  Ep  6 | Loss 5.7406 | Val 11.29% | SW 0.607 | LR 0.00050
  Ep  7 | Loss 5.5871 | Val 12.20% | SW 0.549 | LR 0.00050
  Ep  8 | Loss 5.4008 | Val 12.53% | SW 0.497 | LR 0.00050
  Ep  9 | Loss 5.2749 | Val 14.35% | SW 0.449 | LR 0.00050
  Ep 10 | Loss 5.1764 | Val 14.34% | SW 0.407 | LR 0.00050
 Best: 14.35%

  P1 | 25%
  12500 images | 196 batches/epoch
  Ep  1 | Loss 7.0427 | Val 5.84% | SW 1.000 | LR 0.00050
  Ep  2 | Loss 6.1911 | Val 9.85% | SW 0.905 | LR 0.00050
  Ep  3 | Loss 5.7696 | Val 11.99% | SW 0.819 | LR 0.00050
  Ep  4 | Loss 5.4204 | Val 15.07% | SW 0.741 | LR 0.00050
  Ep  5 | Loss 5.1046 | Val 16.01% | SW 0.670 | LR 0.00050
  Ep  6 | Loss 4